# backward-on-scalar-loss — worked example 2: backward() populates .grad on every leaf in the graph

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-on-scalar-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A single `.backward()` on a scalar loss fills `.grad` on **all** leaf tensors that fed into it. With an affine model `w*x + b`, both `w` and `b` are leaves, so one backward call gives you `w.grad` and `b.grad` simultaneously. The mean reduction divides the summed gradient by N.

## Worked solution

1. **Forward pass.** `pred = w * x + b` broadcasts the scalar params over the input vector `x`, giving a prediction per sample.
2. **Per-sample loss then reduce.** `loss = ((pred - y) ** 2).mean()` first builds a vector of squared errors, then `.mean()` reduces it to a scalar `L = (1/N) \sum_i (w x_i + b - y_i)^2`. The reduction is mandatory so `.backward()` has a scalar to differentiate.
3. **One backward, two grads.** `loss.backward()` traverses the autograd graph back to both leaves. Autograd accumulates `dL/dw` into `w.grad` and `dL/db` into `b.grad` in the same pass — you do not call backward twice.
4. **Why the grads are what they are.** With the `1/N` from `.mean()`, `dL/dw = (2/N) \sum_i (pred_i - y_i) x_i` and `dL/db = (2/N) \sum_i (pred_i - y_i)`. The `b` gradient drops the `x_i` factor because `d(pred_i)/db = 1`.
5. **Return both grads** so the caller can confirm autograd populated every leaf, not just `w`.

In [ ]:
def backward_two_leaves(w, b, x, y):
    pred = w * x + b
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    return w.grad, b.grad

t.manual_seed(0)
w = t.tensor([1.5], requires_grad=True)
b = t.tensor([0.5], requires_grad=True)
x = t.tensor([2.0, 4.0, 6.0])
y = t.tensor([3.0, 5.0, 8.0])
gw, gb = backward_two_leaves(w, b, x, y)
print('w.grad:', gw)
print('b.grad:', gb)
n = x.numel()
pred = 1.5 * x + 0.5
print('expected w.grad:', ((2 / n) * (pred - y) * x).sum().item())
print('expected b.grad:', ((2 / n) * (pred - y)).sum().item())